# state_dict : recharger un modèle, puis reprendre Adam

Savoir ce qu’un fichier permet réellement de retrouver.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#sauvegarder-et-reprendre).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Nous voulons retrouver demain la règle qui transformait nos deux descripteurs en scores. Commençons par sauvegarder un modèle simple, sans historique d’apprentissage.

La couche reprend les poids choisis à la main dans le socle : seule la seconde mesure agit, avec les poids 1 et −1. Son biais est nul.

| Nom | Forme | Valeurs |
| --- | --- | --- |
| weight | (2,2) | [[0,1],[0,−1]] |
| bias | (2,) | [0,0] |

state_dict est un dictionnaire de tenseurs nommés. Il ne remplace pas la définition de la classe ou de l’architecture.

Le programme écrit modele-simple.pt dans son dossier courant. Pour le relire, nous recréons une couche Linear(2,2), puis chargeons ses valeurs.

## Démonstration 1 · Sauvegarder la règle et refaire la même prédiction

Comparer les scores, pas seulement l’existence du fichier.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
from pathlib import Path
import torch
from torch import nn


### Calculer l’état suivant

no_grad permet de choisir les poids sans enregistrer ce réglage manuel. copy_ écrit les valeurs ; zero_ annule les biais.

In [4]:
modele = nn.Linear(2, 2)


### Appliquer le contexte explicite

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [5]:
with torch.no_grad():
    modele.weight.copy_(torch.tensor([[0., 1.], [0., -1.]]))
    modele.bias.zero_()


### Calculer l’état suivant

La ligne [0,2 ; 4,5] représente le deuxième extrait fictif. Le modèle produit les scores [4,5 ; −4,5].

In [6]:
entree = torch.tensor([[0.2, 4.5]])
modele.eval()


Linear(in_features=2, out_features=2, bias=True)

In [7]:
observer('entree', entree)

entree forme (1, 2) type torch.float32 requires_grad False
  valeurs : [[0.20000000298023224, 4.5]]


### Appliquer le contexte explicite

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [8]:
with torch.no_grad():
    scores_avant = modele(entree)


### Calculer l’état suivant

torch.save sérialise les poids et biais dans le fichier choisi. Les noms des classes voix/percussion restent une convention extérieure à ce petit fichier.

In [9]:
fichier = Path("modele-simple.pt")
torch.save(modele.state_dict(), fichier)



In [10]:
observer('fichier', fichier)

### Calculer l’état suivant

La nouvelle couche possède la même architecture. map_location="cpu" place les tenseurs chargés sur CPU ; weights_only=True limite le chargement aux types attendus ici.

Les clés weight et bias indiquent où remettre les deux tenseurs. Leurs formes doivent correspondre à cette couche.

In [11]:
copie = nn.Linear(2, 2)
etat = torch.load(fichier, map_location="cpu", weights_only=True)
copie.load_state_dict(etat)


<All keys matched successfully>

In [12]:
observer('copie', copie)
observer('etat', etat)

etat : clés ['weight', 'bias']


### Calculer l’état suivant

load_state_dict ne sélectionne pas eval. Nous le demandons, puis comparons les scores sur la même entrée avec torch.equal.

In [13]:
copie.eval()


Linear(in_features=2, out_features=2, bias=True)

### Appliquer le contexte explicite

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [14]:
with torch.no_grad():
    scores_apres = copie(entree)


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [15]:
print("cles :", list(etat))
print("scores avant :", scores_avant.tolist())
print("scores apres :", scores_apres.tolist())
print("identiques :", torch.equal(scores_avant, scores_apres))

cles : ['weight', 'bias']
scores avant : [[4.5, -4.5]]
scores apres : [[4.5, -4.5]]
identiques : True


Le fichier permet de retrouver ces valeurs ; l’architecture et la préparation doivent aussi rester identifiables.

### Reprendre un apprentissage avec Adam

Adam conserve aussi des moyennes de gradients et un compteur. Si nous voulons poursuivre, les seuls poids du modèle ne restituent pas cet historique.

Cette seconde version entraîne une fois une Linear(2,2) sur les trois exemples fictifs. Le fichier reprise-adam.pt contient modèle, optimiseur et repère d’époque.

## Démonstration 2 · Charger les deux états dans les bons objets

Recréer le modèle et l’optimiseur qui le référence avant leur chargement.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [16]:
from pathlib import Path
import torch
from torch import nn


### Calculer l’état suivant

Les entrées ont la forme (3,2), avec une classe par ligne. Adam reçoit la matrice de poids et le vecteur de biais de la couche.

In [17]:
torch.manual_seed(17)
entrees = torch.tensor([[0.1, 1.5], [0.2, 4.5], [0.15, 2.5]])
cibles = torch.tensor([0, 1, 0], dtype=torch.long)
modele = nn.Linear(2, 2)


In [18]:
observer('entrees', entrees)
observer('cibles', cibles)

entrees forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[0.10000000149011612, 1.5], [0.20000000298023224, 4.5], [0.15000000596046448, 2.5]]
cibles forme (3,) type torch.int64 requires_grad False
  valeurs : [0, 1, 0]


### Construire la règle de perte

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [19]:
optimiseur = torch.optim.Adam(modele.parameters(), lr=0.01)
critere = nn.CrossEntropyLoss()


### Remettre les gradients à zéro

zero_grad précède le calcul des scores et de la perte. backward fournit les gradients ; step crée les états Adam de ces deux tenseurs de paramètres.

In [20]:
optimiseur.zero_grad()


In [21]:
observer_entrainement({'entrees': entrees, 'cibles': cibles, 'modele': modele, 'optimiseur': optimiseur, 'critere': critere}, montrer_poids=False)

modele.weight.grad = None
modele.bias.grad = None


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [22]:
scores = modele(entrees)
perte = critere(scores, cibles)


In [23]:
observer('scores', scores)
observer('perte', perte)

scores forme (3, 2) type torch.float32 requires_grad True
  valeurs : [[-0.6004615426063538, -0.6813506484031677], [-0.4608038663864136, -2.230463743209839], [-0.5554589033126831, -1.1899385452270508]]
perte forme () type torch.float32 requires_grad True
  valeurs : 1.0019747018814087


### Calculer les gradients

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [24]:
perte.backward()


In [25]:
observer_entrainement({'entrees': entrees, 'cibles': cibles, 'modele': modele, 'optimiseur': optimiseur, 'critere': critere, 'scores': scores, 'perte': perte}, montrer_poids=False)

modele.weight.grad forme (2, 2) type torch.float32 requires_grad False
  valeurs : [[0.023643290624022484, 0.7529823184013367], [-0.023643292486667633, -0.7529823780059814]]
modele.bias.grad forme (2,) type torch.float32 requires_grad False
  valeurs : [0.009377002716064453, -0.009377047419548035]


### Appliquer une mise à jour

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [26]:
optimiseur.step()


In [27]:
observer_entrainement({'entrees': entrees, 'cibles': cibles, 'modele': modele, 'optimiseur': optimiseur, 'critere': critere, 'scores': scores, 'perte': perte}, montrer_poids=True)

modele.weight forme (2, 2) type torch.float32 requires_grad True
  valeurs : [[-0.10299695283174515, 0.03965245932340622], [0.4769856333732605, -0.5219372510910034]]
modele.weight.grad forme (2, 2) type torch.float32 requires_grad False
  valeurs : [[0.023643290624022484, 0.7529823184013367], [-0.023643292486667633, -0.7529823780059814]]
modele.bias forme (2,) type torch.float32 requires_grad True
  valeurs : [-0.6756405234336853, 0.07985663414001465]
modele.bias.grad forme (2,) type torch.float32 requires_grad False
  valeurs : [0.009377002716064453, -0.009377047419548035]


### Calculer l’état suivant

model et optimizer nomment deux contenus différents. epoch vaut 1 parce que ce programme vient de réaliser son unique passage.

In [28]:
fichier = Path("reprise-adam.pt")
torch.save({"model": modele.state_dict(),
            "optimizer": optimiseur.state_dict(), "epoch": 1}, fichier)


In [29]:
observer('fichier', fichier)

### Calculer l’état suivant

Le nouvel optimiseur reçoit les paramètres de copie. Son chargement ajoute les moments et compteurs sauvegardés, au lieu de repartir de zéro.

In [30]:
etat = torch.load(fichier, map_location="cpu", weights_only=True)
copie = nn.Linear(2, 2)
nouvel_optimiseur = torch.optim.Adam(copie.parameters(), lr=0.01)
copie.load_state_dict(etat["model"])


<All keys matched successfully>

In [31]:
observer('etat', etat)
observer('copie', copie)
observer('nouvel_optimiseur', nouvel_optimiseur)

etat : clés ['model', 'optimizer', 'epoch']


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [32]:
nouvel_optimiseur.load_state_dict(etat["optimizer"])


### Lire les sorties

Deux entrées d’état correspondent aux deux tenseurs de paramètres, pas à deux neurones. L’égalité des poids confirme le rechargement.

In [33]:
print("etats Adam :", len(nouvel_optimiseur.state))
print("epoque :", etat["epoch"])
print("poids identiques :", torch.equal(modele.weight, copie.weight))

etats Adam : 2
epoque : 1
poids identiques : True


Reprendre l’optimisation demande les états actifs, pas seulement une architecture qui accepte les poids.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Après sauvegarde et rechargement de ce Linear, quelle sortie obtient-on pour x=1 ?

In [34]:
exo_modele = torch.nn.Linear(1,1)
with torch.no_grad():
    exo_modele.weight.fill_(2.)
    exo_modele.bias.fill_(1.)
torch.save(exo_modele.state_dict(),"exo-modele.pt")
exo_recharge = torch.nn.Linear(1,1)
exo_recharge.load_state_dict(torch.load("exo-modele.pt",weights_only=True))
exo_sortie = ...  # Remplacer les points par votre réponse.
if exo_sortie is Ellipsis or exo_sortie is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_sortie)-exo_recharge(torch.ones(1,1)).item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

La sauvegarde garde ici poids et biais : 2×1+1.

</details>

### Exercice 2

Un nouveau Linear sans chargement est-il nécessairement identique au modèle précédent ?

In [35]:
exo_a = torch.nn.Linear(1,1)
exo_b = torch.nn.Linear(1,1)
with torch.no_grad():
    exo_a.weight.fill_(2.)
    exo_b.weight.fill_(0.)
exo_identique = ...  # Remplacer les points par votre réponse.
if exo_identique is Ellipsis or exo_identique is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_identique is torch.equal(exo_a.weight,exo_b.weight)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Créer la même architecture ne recopie pas ses paramètres. Les valeurs ont été explicitement rendues différentes ici.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_modele = torch.nn.Linear(1,1)
with torch.no_grad():
    exo_modele.weight.fill_(2.)
    exo_modele.bias.fill_(1.)
torch.save(exo_modele.state_dict(),"exo-modele.pt")
exo_recharge = torch.nn.Linear(1,1)
exo_recharge.load_state_dict(torch.load("exo-modele.pt",weights_only=True))
exo_sortie = 3.
if exo_sortie is Ellipsis or exo_sortie is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_sortie)-exo_recharge(torch.ones(1,1)).item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_a = torch.nn.Linear(1,1)
exo_b = torch.nn.Linear(1,1)
with torch.no_grad():
    exo_a.weight.fill_(2.)
    exo_b.weight.fill_(0.)
exo_identique = False
if exo_identique is Ellipsis or exo_identique is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_identique is torch.equal(exo_a.weight,exo_b.weight)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [36]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()